In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)

VOL = "/Volumes/workspace/default/maplebank"

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])

# For dims, schema-on-read with just the columns we need today
df_txn      = spark.read.option("header", True).schema(txn_schema).csv(f"{VOL}/fact_transactions.csv")
df_customer = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOL}/dim_customer.csv")
df_branch   = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOL}/dim_branch.csv")

print(f"Loaded: {df_txn.count():,} txns, {df_customer.count():,} customers, {df_branch.count()} branches")

Loaded: 10,000 txns, 1,000 customers, 15 branches


In [0]:
# Large transactions: FINTRAC LCTR territory (≥ CAD $10,000)
df_large = df_txn.filter(F.col("amount_cad") >= 10000)

# Multiple conditions — & not 'and', parentheses MANDATORY
df_large_etransfer = df_txn.filter(
    (F.col("transaction_type") == "INTERAC_ETRANSFER") &
    (F.col("amount_cad") >= 10000)
)

print(f"All large txns (≥$10K):        {df_large.count():>4}")
print(f"Large Interac e-Transfers:     {df_large_etransfer.count():>4}")

df_large_etransfer.select("transaction_id", "customer_id", "amount_cad", "transaction_date").show(5)

All large txns (≥$10K):         303
Large Interac e-Transfers:       40
+--------------+-----------+----------+----------------+
|transaction_id|customer_id|amount_cad|transaction_date|
+--------------+-----------+----------+----------------+
|    TXN1000490| CUST100215|  29896.94|      2025-11-13|
|    TXN1000766| CUST100824|  34842.32|      2025-11-24|
|    TXN1000819| CUST100681|   14504.8|      2025-11-03|
|    TXN1001750| CUST100728|  30663.45|      2025-11-07|
|    TXN1001816| CUST100031|  27648.76|      2025-11-12|
+--------------+-----------+----------+----------------+
only showing top 5 rows


In [0]:
df_flagged = df_txn \
    .withColumn("hst_13pct", F.round(F.col("amount_cad") * 0.13, 2)) \
    .withColumn(
        "fintrac_flag",
        F.when(F.col("amount_cad") >= 10000, "LCTR_REPORTABLE")
         .when(F.col("amount_cad") >= 9000,  "STRUCTURING_RISK")   # just-below-threshold = AML red flag
         .otherwise("NORMAL")
    ) \
    .withColumn(
        "value_band",
        F.when(F.col("amount_cad") < 100,   "MICRO")
         .when(F.col("amount_cad") < 1000,  "SMALL")
         .when(F.col("amount_cad") < 10000, "MEDIUM")
         .otherwise("LARGE")
    )

df_flagged.groupBy("fintrac_flag").count().orderBy("fintrac_flag").show()
df_flagged.select("transaction_id", "amount_cad", "fintrac_flag", "value_band").show(8)

+---------------+-----+
|   fintrac_flag|count|
+---------------+-----+
|LCTR_REPORTABLE|  303|
|         NORMAL| 9697|
+---------------+-----+

+--------------+----------+------------+----------+
|transaction_id|amount_cad|fintrac_flag|value_band|
+--------------+----------+------------+----------+
|    TXN1000001|   3625.93|      NORMAL|    MEDIUM|
|    TXN1000002|   4765.46|      NORMAL|    MEDIUM|
|    TXN1000003|   4136.76|      NORMAL|    MEDIUM|
|    TXN1000004|   2222.88|      NORMAL|    MEDIUM|
|    TXN1000005|   3303.71|      NORMAL|    MEDIUM|
|    TXN1000006|   4466.38|      NORMAL|    MEDIUM|
|    TXN1000007|   4329.38|      NORMAL|    MEDIUM|
|    TXN1000008|   4690.19|      NORMAL|    MEDIUM|
+--------------+----------+------------+----------+
only showing top 8 rows


In [0]:
# Inner join: only transactions with a known customer
df_enriched = df_flagged.join(
    df_customer.select("customer_id", "first_name", "last_name", "city", "province"),
    on="customer_id",
    how="inner"
).join(
    df_branch.select("branch_id", "branch_name"),
    on="branch_id",
    how="left"      # left: keep the txn even if branch lookup fails
)

print(f"Before joins: {df_flagged.count():,}   After: {df_enriched.count():,}")

# A FINTRAC analyst's view: who's behind the large transactions?
df_enriched.filter(F.col("fintrac_flag") == "LCTR_REPORTABLE") \
    .select("transaction_id", "first_name", "last_name", "city", "province",
            "amount_cad", "transaction_type", "branch_name") \
    .orderBy(F.col("amount_cad").desc()) \
    .show(10, truncate=False)

Before joins: 10,000   After: 10,000
+--------------+----------+---------+-----------+--------+----------+-----------------+----------------------------+
|transaction_id|first_name|last_name|city       |province|amount_cad|transaction_type |branch_name                 |
+--------------+----------+---------+-----------+--------+----------+-----------------+----------------------------+
|TXN1003323    |Anthony   |White    |Victoria   |BC      |49833.93  |POS_PURCHASE     |MapleBank Toronto Branch    |
|TXN1002241    |Justin    |Gregory  |Halifax    |NS      |49698.29  |EFT              |MapleBank Hamilton Branch   |
|TXN1007365    |Holly     |Stevenson|Montreal   |QC      |49203.64  |INTERAC_ETRANSFER|MapleBank Mississauga Branch|
|TXN1006071    |Lisa      |Andrews  |Winnipeg   |MB      |49201.98  |BILL_PAYMENT     |MapleBank Calgary Branch    |
|TXN1002375    |Gloria    |Thomas   |Quebec City|QC      |49056.2   |ATM_WITHDRAWAL   |MapleBank Toronto Branch    |
|TXN1004087    |Jaime     |

In [0]:
# Transactions whose customer_id does NOT exist in dim_customer
df_orphans = df_txn.join(df_customer, on="customer_id", how="left_anti")
orphans = df_orphans.count()

print(f"Orphan transactions: {orphans}")
if orphans == 0:
    print("✅ Referential integrity clean — every txn has a known customer")
else:
    df_orphans.select("transaction_id", "customer_id", "amount_cad").show(10)

Orphan transactions: 0
✅ Referential integrity clean — every txn has a known customer


In [0]:
# Ontario branches ranked by Interac e-Transfer volume
df_enriched \
    .filter(F.col("transaction_type") == "INTERAC_ETRANSFER") \
    .filter(F.col("province") == "ON") \
    .groupBy("branch_id", "branch_name") \
    .agg(
        F.count("*").alias("txn_count"),
        F.round(F.sum("amount_cad"), 2).alias("total_cad"),
        F.round(F.avg("amount_cad"), 2).alias("avg_cad"),
        F.countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy(F.col("total_cad").desc()) \
    .show(5, truncate=False)

+----------+----------------------------+---------+---------+-------+----------------+
|branch_id |branch_name                 |txn_count|total_cad|avg_cad|unique_customers|
+----------+----------------------------+---------+---------+-------+----------------+
|BR_OTT_003|MapleBank Ottawa Branch     |18       |161650.08|8980.56|13              |
|BR_CAL_012|MapleBank Calgary Branch    |30       |119816.56|3993.89|19              |
|BR_HAM_004|MapleBank Hamilton Branch   |9        |62218.46 |6913.16|8               |
|BR_HAL_015|MapleBank Halifax Branch    |13       |59495.92 |4576.61|9               |
|BR_QUE_007|MapleBank Quebec City Branch|20       |54686.26 |2734.31|16              |
+----------+----------------------------+---------+---------+-------+----------------+
only showing top 5 rows
